# 07 — DADPc (Domain Adaptive Deep Possibilistic Clustering)

**Family:** Domain adaptation &nbsp;•&nbsp; **Input:** DE features `[14 x 5]`

DADPc targets the **subject-independent** protocol. A shared feature extractor maps
DE features to an embedding; a linear head classifies the labelled source (22
training subjects). The cross-subject domain gap is reduced by a **possibilistic
clustering objective with a fuzzy-entropy regularizer** in embedding space, plus a
light multi-kernel MMD term:

`L = CE(source) + w_mmd·MMD(source, target) + w_pc·PCM(all) + w_fe·fuzzy_entropy(memberships)`

`target` = the **unlabelled** left-out subject (transductive domain adaptation —
its windows are visible during training, its labels never are).

## 1. Setup

In [ ]:
import os, sys, time, json
from functools import partial

import numpy as np
import torch

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.evaluation import (
    set_seed, get_device, count_parameters,
    make_default_fit, default_predict,
    run_subject_dependent, run_loso, save_results, quick_report,
)
from src.data_loader import (
    build_subject_dependent_loaders, build_loso_loaders, load_subject, REDUCED_5_CHANNELS,
)

set_seed(42)
DEVICE = get_device()

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
SD_SPLITS     = os.path.join(ROOT, "data", "splits", "subject_dependent_splits.json")
LOSO_SPLITS   = os.path.join(ROOT, "data", "splits", "loso_splits.json")
RESULTS_DIR   = os.path.join(ROOT, "results")

# ----- experiment configuration -----
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> a 3-subject / 3-fold smoke test with few epochs
                      # False -> the full benchmark used for the thesis table
print("device:", DEVICE, "| label:", LABEL, "| QUICK:", QUICK)

import torch.nn as nn
from src.evaluation import compute_metrics, summarize_per_fold
from src.models import dadpc
MODEL_NAME = "DADPc"

## 2. Architecture

`src/models/dadpc.py`: `FeatureExtractor` (3-layer MLP with BN/dropout, 70 → 64-D
embedding) + linear classifier + `K = n_classes` learnable cluster centers with a
learnable per-cluster bandwidth `eta`. `memberships` implements the possibilistic
`u_ik = 1 / (1 + (d²/eta)^{1/(m-1)})` (not simplex-constrained). `dadpc_loss`
assembles the four terms. See the `# ASSUMPTION` notes in the source.

In [ ]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (14, 5)}
_m = dadpc.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

## 3. Subject-independent (LOSO) protocol — the primary result

DADPc needs the target subject's (unlabelled) windows during training, which the
generic `run_loso` harness does not pass to `fit_fn`. So this protocol uses a
bespoke loop; metrics still go through `compute_metrics` / `summarize_per_fold` for
schema parity with every other notebook.

In [ ]:
W_MMD, W_PC, W_FE = 1.0, 0.1, 0.1
EPOCHS_LOSO = 8 if QUICK else 60
LR = 1e-3

with open(LOSO_SPLITS) as f:
    loso = json.load(f)
folds = list(range(3)) if QUICK else list(range(loso["n_folds"]))

per_fold = []
t_total = 0.0
params_count = None

for k in folds:
    tl, vl, el, meta = build_loso_loaders(
        PROCESSED_DIR, LOSO_SPLITS, fold=k, label=LABEL, feature="de",
        batch_size=256, normalize=True, val_frac=0.0,
    )
    # target (unlabelled) windows for this fold
    tgt_X = torch.tensor(np.stack([el.dataset[i][0].numpy() for i in range(len(el.dataset))]),
                         dtype=torch.float32)

    model = dadpc.build({"n_classes": 2, "meta": meta}).to(DEVICE)
    if params_count is None:
        params_count = count_parameters(model)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

    t0 = time.time()
    for ep in range(EPOCHS_LOSO):
        model.train()
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            # sample a target minibatch of matching size
            idx = torch.randint(0, tgt_X.shape[0], (xb.shape[0],))
            xt = tgt_X[idx].to(DEVICE)
            opt.zero_grad()
            loss, parts = model.dadpc_loss(xb, yb, xt, W_MMD, W_PC, W_FE)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
    t_total += time.time() - t0

    model.eval()
    with torch.no_grad():
        logits = model(tgt_X.to(DEVICE))
        pred = logits.argmax(1).cpu().numpy()
    yte = np.array([int(el.dataset[i][1]) for i in range(len(el.dataset))])
    m = compute_metrics(yte, pred); m["fold"] = k; m["test_subject"] = meta["test_subject"]
    per_fold.append(m)
    print(f"  fold {k:02d} ({meta['test_subject']}): acc={m['accuracy']:.3f} f1={m['f1']:.3f}")

loso_res = summarize_per_fold(per_fold, params_count, t_total, protocol="loso")
quick_report(MODEL_NAME, loso_res)

## 4. Subject-dependent protocol (reference)

Within a single subject there is no cross-subject domain gap, so DADPc reduces to
its supervised backbone. We run it here (CE only, via the shared harness) purely so
the consolidated table has a comparable subject-dependent number.

In [ ]:
FEATURE, CHANNELS = "de", None
BATCH_SD, EPOCHS_SD = 64, 80

sd_loader = partial(
    build_subject_dependent_loaders,
    processed_dir=PROCESSED_DIR, splits_path=SD_SPLITS,
    label=LABEL, feature=FEATURE, batch_size=BATCH_SD, channels=CHANNELS, normalize=True,
)
sd_subjects = ["subject_01", "subject_02", "subject_03"] if QUICK else None

sd_res = run_subject_dependent(
    build_model=dadpc.build, processed_dir=PROCESSED_DIR, splits_path=SD_SPLITS,
    loader_factory=sd_loader,
    fit_fn=make_default_fit(epochs=(5 if QUICK else EPOCHS_SD), lr=LR),
    predict_fn=default_predict, device=DEVICE,
    model_kwargs={"n_classes": 2}, subjects=sd_subjects,
)
quick_report(MODEL_NAME, sd_res)

## 5. Save results

In [ ]:
save_results(
    MODEL_NAME, subject_dependent=sd_res, loso=loso_res,
    family="Domain adaptation", results_dir=RESULTS_DIR,
    config={"label": LABEL, "feature": "de", "w_mmd": W_MMD, "w_pc": W_PC,
            "w_fe": W_FE, "epochs_loso": EPOCHS_LOSO, "epochs_sd": EPOCHS_SD,
            "lr": LR, "quick": QUICK},
)

## Notes for the methodology chapter

* This is the one model whose analysis should **lead with LOSO**; the
  subject-dependent number is a backbone-only reference.
* Loss weights `w_mmd / w_pc / w_fe` are the main knobs — a small sweep belongs in
  the appendix.
* Transductive setup: the left-out subject's raw windows (never labels) are used
  during training, which is standard for domain-adaptation EEG papers but should be
  stated explicitly in the thesis.